# Milestone 1: Automated High-Throughput Pipeline (Ultra-Fast Edition)
**Pipeline cào dữ liệu tối ưu tốc độ cao cho Google Colab CPU-only:**
- ⚡ **Local SSD Cache (`/content/fast_arxiv_data/`):** Loại bỏ hoàn toàn nghẽn Google Drive FUSE I/O.
- 🚀 **Zero-Second Skip & In-Memory Manifest Indexing O(1):** Không quét đệ quy rglob/stat trên đĩa, nạp trạng thái trong 0.002s.
- 🔄 **Interleaved Overlapped Pipelining:** Tận dụng triệt để thời gian nghỉ 3.0s của arXiv để giải nén CPU và cào Semantic Scholar Batch API.
- 🤝 **Multi-Colab Sharding:** Hỗ trợ chạy song song 2-4 máy Colab chia sẻ dải ID không trùng lặp, tăng tốc tuyến tính.
- 📊 **Single Stats File:** Xuất đúng 1 file thống kê duy nhất `crawl_session_stats.json` & `.csv` chứa đủ 7 chỉ số benchmark cho báo cáo Jupyter Book.


In [ ]:
# KẾT NỐI DRIVE, THIẾT LẬP FAST LOCAL SSD CACHE VÀ NẠP MÃ NGUỒN
import os, sys, subprocess, shutil, time, importlib
from pathlib import Path

# 1. Phát hiện môi trường Colab & Mount Google Drive
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    CODE_DIR = Path("/content/arxiv-code")
    PERSISTENT_DRIVE_DATA = Path(os.environ.get(
        "ARXIV_DATA_DIR", "/content/drive/MyDrive/Colab_Notebooks/arXiv-Project/data"
    )).resolve()
    FAST_LOCAL_DATA = Path("/content/fast_arxiv_data").resolve()
    
    # Clone hoặc pull mã nguồn nếu chưa có ở /content
    BRANCH = "feat/experiment-speed"
    if not (CODE_DIR / ".git").is_dir():
        subprocess.run([
            "git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH,
            "https://github.com/ahtuan171/arXiv-Project.git", str(CODE_DIR),
        ], check=False)
        if not (CODE_DIR / "src").is_dir():
            CODE_DIR = Path.cwd().resolve()
    else:
        subprocess.run(["git", "-C", str(CODE_DIR), "pull", "origin", BRANCH], check=False)
    os.chdir(CODE_DIR)

    # Cài đặt tự động các thư viện phụ thuộc (arxiv, httpx, ...) nếu chưa có trên Colab
    req_file = CODE_DIR / "requirements.txt"
    if req_file.is_file():
        print("📦 Đang kiểm tra & cài đặt dependencies từ requirements.txt...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req_file)], check=False)
        importlib.invalidate_caches()
        print("✅ Dependencies (arxiv, httpx, pandas, ...) đã sẵn sàng.")
else:
    CODE_DIR = Path.cwd().resolve()
    PERSISTENT_DRIVE_DATA = Path(os.environ.get("ARXIV_DATA_DIR", CODE_DIR / "data")).resolve()
    FAST_LOCAL_DATA = (CODE_DIR / "fast_arxiv_data").resolve()

# 2. Khởi tạo cấu trúc thư mục SSD cực nhanh
FAST_LOCAL_DATA.mkdir(parents=True, exist_ok=True)
(FAST_LOCAL_DATA / "raw").mkdir(exist_ok=True)
(FAST_LOCAL_DATA / "csv").mkdir(exist_ok=True)

# Gán biến môi trường trỏ toàn bộ pipeline vào Local SSD
os.environ["ARXIV_DATA_DIR"] = str(FAST_LOCAL_DATA)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Cập nhật src.config nếu đã được nạp
if "src.config" in sys.modules:
    from src import config
    config.DEFAULT_DATA_DIR = FAST_LOCAL_DATA
    config.DATA_DIR = FAST_LOCAL_DATA

# 3. Fast Snapshot Bootstrap (< 2 giây): Chỉ copy file metadata JSONL & CSV từ Drive sang SSD
# TUYỆT ĐỐI KHÔNG copy thư mục raw/ để tránh làm chậm khởi động!
if PERSISTENT_DRIVE_DATA.is_dir():
    t0 = time.perf_counter()
    copied_count = 0
    for pattern in ("*.jsonl*", "*.json"):
        for f in PERSISTENT_DRIVE_DATA.glob(pattern):
            if f.is_file():
                shutil.copy2(f, FAST_LOCAL_DATA / f.name)
                copied_count += 1
    if (PERSISTENT_DRIVE_DATA / "csv").is_dir():
        for f in (PERSISTENT_DRIVE_DATA / "csv").glob("*.csv"):
            if f.is_file():
                shutil.copy2(f, FAST_LOCAL_DATA / "csv" / f.name)
                copied_count += 1
    elapsed = time.perf_counter() - t0
    print(f"✅ Đã nạp nhanh {copied_count} files snapshot metadata từ Drive vào Local SSD ({elapsed:.2f}s).")
else:
    print("ℹ️ Chưa có thư mục Drive cũ; khởi tạo Local SSD sạch hoàn toàn.")

print(f"🚀 Thư mục làm việc tốc độ cao: {FAST_LOCAL_DATA}")
print(f"💾 Thư mục đồng bộ Drive: {PERSISTENT_DRIVE_DATA}")


In [ ]:
#@title ⚙️ BỘ CẤU HÌNH TOÀN DIỆN THU THẬP TỐI ƯU MILESTONE 1 { display-mode: "form" }

#@markdown ---
#@markdown ### 📌 1. ĐỊNH DANH VÀ DẢI BÀI BÁO (RANGE & TARGET)
start_id = "2412.03848"  #@param {type:"string"}
end_id = "2412.08847"  #@param {type:"string"}
LIMIT = 0  #@param {type:"integer"} # 0 = toàn bộ dải

#@markdown ---
#@markdown ### ⚡ 2. PHÂN CHIA TẢI ĐA MÁY (MULTI-COLAB SHARDING)
#@markdown *Giúp 2-4 máy Colab chạy song song mà không trùng lặp bài báo*
ENABLE_SHARDING = False  #@param {type:"boolean"}
WORKER_INDEX = 0  #@param {type:"integer"} # Chỉ số máy này (0, 1, 2,...)
TOTAL_WORKERS = 4  #@param {type:"integer"} # Tổng số máy Colab cùng cào

#@markdown ---
#@markdown ### 🚀 3. CÔNG TẮC BẬT/TẮT TỪNG GIAI ĐOẠN (PIPELINE STAGE TOGGLES)
STAGE_1_METADATA = True  #@param {type:"boolean"} # Cào Metadata (arXiv API)
STAGE_2_DOWNLOAD_SOURCES = True  #@param {type:"boolean"} # Tải Full-text TeX Sources
STAGE_3_RAW_REFERENCES = True  #@param {type:"boolean"} # Cào References (Semantic Scholar)
STAGE_4_BENCHMARK = True  #@param {type:"boolean"} # Đo đạc RAM/CPU/Disk
STAGE_5_FINAL_AUDIT = False  #@param {type:"boolean"} # Deep Hash & Reconcile toàn vẹn

#@markdown ---
#@markdown ### 🛡️ 4. CHIẾN LƯỢC BỎ QUA & THỬ LẠI (SKIP & RESUME POLICY)
RESUME_MODE = True  #@param {type:"boolean"} # Bỏ qua ngay lập tức bài đã có (Zero-Second Skip)
RETRY_ERRORS = False  #@param {type:"boolean"} # Thử lại các bài bị lỗi mạng/5xx
RETRY_NOT_FOUND = False  #@param {type:"boolean"} # Thử lại các bài 404
RETRY_PARTIAL = False  #@param {type:"boolean"} # Thử lại các bài thiếu version

#@markdown ---
#@markdown ### 💾 5. TỐI ƯU HÓA LƯU TRỮ VÀ I/O (STORAGE ACCELERATION)
CUSTOM_DRIVE_DATA_DIR = "/content/drive/MyDrive/Colab_Notebooks/arXiv-Project/data"  #@param {type:"string"}
AUTO_SHARD_DRIVE_FOLDER = True  #@param {type:"boolean"} # Tự động thêm _worker_X vào thư mục Drive khi bật sharding để tránh đè file
USE_FAST_SCRATCH = True  #@param {type:"boolean"} # Tải và giải nén trên Local SSD
DRIVE_SYNC_INTERVAL_MIN = 15  #@param {type:"integer"} # Phút đồng bộ ngầm sang Drive
AUTO_SYNC_ON_FINISH = True  #@param {type:"boolean"} # Luôn đồng bộ khi hoàn tất phiên

#@markdown ---
#@markdown ### 🌐 6. TINH CHỈNH MẠNG VÀ TỐC ĐỘ (NETWORK & CONCURRENCY)
METADATA_BATCH_SIZE = 100  #@param {type:"integer"} # 100-200 ID/req
RAW_REF_BATCH_SIZE = 100  #@param {type:"integer"} # S2 Batch API (100 an toàn dưới 10MB)
ARXIV_MIN_INTERVAL = 3.0  #@param {type:"number"} # Tối thiểu 3.0s theo luật arXiv
HTTP_PROXY = ""  #@param {type:"string"} # Proxy HTTP/SOCKS5 nếu có
S2_SECRET_NAME = "S2_API_KEY"  #@param {type:"string"} # Secret name trong Colab

#@markdown ---
#@markdown ### 🔑 7. SEMANTIC SCHOLAR API & CHỐNG RATE LIMIT (S2 SETTINGS)
#@markdown *Nhập key tại đây nếu không dùng Colab Secret; tăng interval lên 2.0-3.0s nếu bị 429*
S2_API_KEY_INPUT = ""  #@param {type:"string"} # Nhập key s2k-... vào đây nếu có
S2_MIN_INTERVAL = 2.0  #@param {type:"number"} # Giây giữa các request S2 (khuyến nghị >= 2.0s)
S2_RATE_LIMIT_COOLDOWN_SEC = 25.0  #@param {type:"number"} # Giây tạm nghỉ khi gặp HTTP 429 để hồi quota

# Thực thi phân chia dải ID
from src.utils import parse_arxiv_range, canonical_arxiv_id
all_ids = parse_arxiv_range(start_id, end_id)
if LIMIT > 0:
    all_ids = all_ids[:LIMIT]

if ENABLE_SHARDING:
    if WORKER_INDEX >= TOTAL_WORKERS or WORKER_INDEX < 0:
        raise ValueError(f"WORKER_INDEX ({WORKER_INDEX}) phải thỏa mãn: 0 <= WORKER_INDEX < TOTAL_WORKERS ({TOTAL_WORKERS})")
    target_ids = [aid for i, aid in enumerate(all_ids) if i % TOTAL_WORKERS == WORKER_INDEX]
    print(f"🔹 [Worker {WORKER_INDEX + 1}/{TOTAL_WORKERS}]: Nhận {len(target_ids)}/{len(all_ids)} bài báo trong dải.")
else:
    target_ids = all_ids
    print(f"🔹 Mục tiêu phiên này: {len(target_ids)} bài báo ({target_ids[0] if target_ids else ''} -> {target_ids[-1] if target_ids else ''}).")

# Áp dụng tùy biến đường dẫn Google Drive
if "CUSTOM_DRIVE_DATA_DIR" in globals() and CUSTOM_DRIVE_DATA_DIR.strip():
    custom_p = Path(CUSTOM_DRIVE_DATA_DIR.strip()).resolve()
    if ENABLE_SHARDING and globals().get("AUTO_SHARD_DRIVE_FOLDER", False):
        PERSISTENT_DRIVE_DATA = custom_p.parent / f"{custom_p.name}_worker_{WORKER_INDEX}"
    else:
        PERSISTENT_DRIVE_DATA = custom_p
    PERSISTENT_DRIVE_DATA.mkdir(parents=True, exist_ok=True)
    (PERSISTENT_DRIVE_DATA / "raw").mkdir(exist_ok=True)
    (PERSISTENT_DRIVE_DATA / "csv").mkdir(exist_ok=True)
    print(f"💾 Thư mục đồng bộ Drive: {PERSISTENT_DRIVE_DATA}")


In [ ]:
# KHỞI TẠO BACKGROUND DRIVE SYNC WORKER VÀ BỘ ĐO HIỆU NĂNG PHIÊN
import queue, threading, time, psutil, requests
from src import config

# 1. Đọc S2 API Key từ Form cấu hình, Colab Secrets hoặc file .env
s2_key = (S2_API_KEY_INPUT if "S2_API_KEY_INPUT" in globals() else "").strip()
if not s2_key:
    try:
        from google.colab import userdata
        s2_key = userdata.get(S2_SECRET_NAME) or userdata.get("s2_api_key") or userdata.get("S2_API_KEY")
    except Exception:
        s2_key = os.environ.get("S2_API_KEY", "")

# Tìm trong file .env nếu có trên Drive/mã nguồn
if not s2_key and "CODE_DIR" in globals() and CODE_DIR:
    env_file = CODE_DIR / ".env"
    if env_file.is_file():
        for line in env_file.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line.startswith("S2_API_KEY="):
                s2_key = line.split("=", 1)[1].strip().strip("'").strip('"')
                break

eff_interval = float(S2_MIN_INTERVAL) if "S2_MIN_INTERVAL" in globals() else 2.0
eff_cooldown = float(S2_RATE_LIMIT_COOLDOWN_SEC) if "S2_RATE_LIMIT_COOLDOWN_SEC" in globals() else 25.0

os.environ["S2_RATE_LIMIT_COOLDOWN"] = str(eff_cooldown)
config.S2_RATE_LIMIT_COOLDOWN = eff_cooldown

if s2_key:
    s2_key = str(s2_key).strip().strip("'").strip('"')
    os.environ["S2_API_KEY"] = s2_key
    config.S2_API_KEY = s2_key
    os.environ["S2_API_MIN_INTERVAL"] = str(eff_interval)
    config.S2_API_MIN_INTERVAL = eff_interval
    masked_key = s2_key[:6] + "..." + s2_key[-4:] if len(s2_key) > 10 else "***"
    print(f"🔑 S2 API Key: Đã nạp thành công [{masked_key}] (Interval: {eff_interval}s, Cooldown 429: {eff_cooldown}s).")
    
    # 🔍 TEST NGAY TRẠNG THÁI KEY: Gửi 1 request thử nghiệm để kiểm tra rate limit
    print("🔍 Đang kiểm tra trạng thái hoạt động và hạn ngạch của S2 API Key...")
    try:
        test_url = "https://api.semanticscholar.org/graph/v1/paper/arXiv:2502.13235?fields=paperId,title"
        test_resp = requests.get(test_url, headers={"x-api-key": s2_key}, timeout=10)
        if test_resp.status_code == 200:
            print("✅ [S2 Health Check]: Key HOẠT ĐỘNG TỐT (HTTP 200 OK) — Sẵn sàng cào không lo rate limit!")
        elif test_resp.status_code == 429:
            retry_after_hdr = test_resp.headers.get("Retry-After") or test_resp.headers.get("retry-after")
            print(f"⚠️ [S2 Health Check]: Key HIỆN ĐANG BỊ RATE LIMIT (HTTP 429)! Cần chờ ~{retry_after_hdr or '25'}s trước khi gọi tiếp.")
        elif test_resp.status_code in (401, 403):
            print(f"❌ [S2 Health Check]: Key KHÔNG HỢP LỆ hoặc HẾT HẠN (HTTP {test_resp.status_code}). Vui lòng kiểm tra lại key!")
        else:
            print(f"ℹ️ [S2 Health Check]: Phản hồi HTTP {test_resp.status_code} ({test_resp.text[:100]})")
    except Exception as exc:
        print(f"⚠️ [S2 Health Check]: Không thể kiểm tra key qua mạng ({exc}). Pipeline vẫn sẽ tiếp tục.")
else:
    os.environ["S2_API_MIN_INTERVAL"] = str(max(eff_interval, 3.0))
    config.S2_API_MIN_INTERVAL = max(eff_interval, 3.0)
    print("⚠️ S2 API Key: Chưa có (Dùng quota ẩn danh - rất dễ bị 429 trên Google Colab IP!).")
    print("💡 Gợi ý: Dán key vào ô S2_API_KEY_INPUT ở Cell 2 hoặc tắt STAGE_3_RAW_REFERENCES nếu không cần trích dẫn.")

# 2. Worker đồng bộ nền sang Google Drive (Non-blocking Dual-Path Sync)
class BackgroundDriveSyncWorker(threading.Thread):
    def __init__(self, local_raw: Path, drive_raw: Path):
        super().__init__(daemon=True)
        self.local_raw = local_raw
        self.drive_raw = drive_raw
        self.queue = queue.Queue()
        self.stopped = False
        self.synced_count = 0

    def enqueue(self, paper_id: str, version: str):
        self.queue.put((paper_id, version))

    def run(self):
        while not self.stopped or not self.queue.empty():
            try:
                task = self.queue.get(timeout=0.5)
            except queue.Empty:
                continue
            paper_id, version = task
            src = self.local_raw / paper_id / version
            dst = self.drive_raw / paper_id / version
            if src.is_dir():
                try:
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    if not dst.exists():
                        shutil.copytree(src, dst, symlinks=False)
                        self.synced_count += 1
                except Exception as exc:
                    pass
            self.queue.task_done()

sync_worker = BackgroundDriveSyncWorker(FAST_LOCAL_DATA / "raw", PERSISTENT_DRIVE_DATA / "raw")
sync_worker.start()
print("⚡ Background Drive Sync Worker đã khởi động (Đồng bộ ngầm trong lúc nghỉ 3.0s của arXiv).")

# 3. Bộ theo dõi tài nguyên & Đếm bài báo chuẩn xác
class FastSessionTracker:
    def __init__(self, target_ids: list[str]):
        self.target_ids = set(target_ids)
        self.start_time = time.time()
        self.process = psutil.Process()
        self.peak_ram_bytes = 0
        self.ram_samples = []
        self.completed_paper_ids = set() # Chứa ID bài báo duy nhất thực tế hoàn thành
        self.downloaded_versions_count = 0
        self.pdf_only_versions_count = 0
        self.raw_refs_total = 0
        self.total_raw_bytes = 0
        self.total_cleaned_bytes = 0
        self.discovery_time_sec = 0.0
        self.download_time_sec = 0.0
        self.reference_time_sec = 0.0

    def sample_ram(self):
        try:
            rss = self.process.memory_info().rss
            self.ram_samples.append(rss)
            if rss > self.peak_ram_bytes:
                self.peak_ram_bytes = rss
            return rss / (1024 * 1024)
        except Exception:
            return 0.0

    def mark_paper_completed(self, paper_id: str):
        self.completed_paper_ids.add(paper_id)

tracker = FastSessionTracker(target_ids)
print("📊 Session Tracker đã sẵn sàng.")

# 4. Khởi động SystemBenchmarkTracker để tự động đo Bảng 4 (m1_version_pipeline_benchmark)
from src.benchmark import SystemBenchmarkTracker
benchmark_tracker = SystemBenchmarkTracker(storage_dirs=[FAST_LOCAL_DATA], disk_sample_interval_sec=0.0)
benchmark_tracker.start()
print("⏱️ System Benchmark Tracker đã kích hoạt (Đo lường Bảng 4).")


In [ ]:
# STAGE 1: KHÁM PHÁ METADATA TỪ ARXIV API (BẢNG 1)
try:
    from src.metadata_module.crawl_papers import crawl_papers
    from src import utils
except ModuleNotFoundError as e:
    missing_mod = getattr(e, "name", str(e))
    print(f"⚠️ Phát hiện thiếu module '{missing_mod}'. Đang tự động cài đặt...")
    import subprocess, sys, importlib
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "arxiv>=2.1", "httpx>=0.27"], check=False)
    importlib.invalidate_caches()
    from src.metadata_module.crawl_papers import crawl_papers
    from src import utils
    print("✅ Đã cài đặt xong thư viện và nạp module thành công!")

t_stage1 = time.perf_counter()
papers_path = FAST_LOCAL_DATA / "papers.jsonl"

if STAGE_1_METADATA:
    # Zero-Second Skip Check: Nạp các ID đã hoàn thành vào RAM
    existing_meta_ids = set()
    if papers_path.is_file() and RESUME_MODE and not RETRY_ERRORS:
        for row in utils.read_jsonl(papers_path):
            aid = row.get("arxiv_id")
            if aid and row.get("title"):
                existing_meta_ids.add(aid)
                existing_meta_ids.add(utils.normalize_arxiv_id(aid))
                existing_meta_ids.add(utils.canonical_arxiv_id(aid))

    meta_todo = [aid for aid in target_ids if aid not in existing_meta_ids]
    
    if not meta_todo:
        print(f"⏩ [Stage 1] Zero-Second Skip: Toàn bộ {len(target_ids)} bài đã có metadata hợp lệ trong papers.jsonl! (0.01s)")
    else:
        print(f"🚀 [Stage 1] Cào metadata cho {len(meta_todo)} bài báo (Batch size: {METADATA_BATCH_SIZE})...")
        meta_res = crawl_papers(
            id_list=meta_todo,
            out_path=papers_path,
            csv_path=FAST_LOCAL_DATA / "csv" / "m1_discovery_stats.csv",
            batch_size=METADATA_BATCH_SIZE,
            resume=RESUME_MODE,
            retry_errors=RETRY_ERRORS,
            retry_not_found=RETRY_NOT_FOUND,
        )
        print("✅ [Stage 1] Hoàn thành cào metadata:", meta_res)

tracker.discovery_time_sec = round(time.perf_counter() - t_stage1, 2)
tracker.sample_ram()


In [ ]:
# STAGE 2: TẢI FULL-TEXT SOURCES THEO PHIÊN BẢN (ARXIV DOWNLOADER)
import pandas as pd
from src.download_module.version_downloader import run_version_downloads

t_stage2 = time.perf_counter()

if STAGE_2_DOWNLOAD_SOURCES:
    print(f"🚀 [Stage 2] Tải LaTeX sources theo version (In-Memory Indexing O(1), Min interval: {ARXIV_MIN_INTERVAL}s)...")
    
    def on_source_event(done, total, event):
        tracker.sample_ram()
        paper = event.get("arxiv_id", "")
        version = event.get("version", "")
        status = event.get("status", "")
        raw_b = event.get("raw_size_bytes", 0)
        clean_b = event.get("cleaned_size_bytes", 0)
        
        if status == "success":
            tracker.downloaded_versions_count += 1
            tracker.total_raw_bytes += raw_b
            tracker.total_cleaned_bytes += clean_b
            sync_worker.enqueue(paper, version)
        elif status == "pdf_only":
            tracker.pdf_only_versions_count += 1
            sync_worker.enqueue(paper, version)
            
        pct = 100 * done / total if total else 100
        print(f"[Source] {done}/{total} ({pct:.1f}%) | {paper}/{version} | {status} (Queue: {sync_worker.queue.qsize()})", flush=True)

    version_result = run_version_downloads(
        FAST_LOCAL_DATA,
        ids=target_ids,
        max_downloads=0,
        dry_run=False,
        min_interval=ARXIV_MIN_INTERVAL,
        check_hashes=False,
        trust_manifest=True, # Dùng index O(1) loại bỏ hoàn toàn FUSE stat
        retry_errors=RETRY_ERRORS,
        retry_not_found=RETRY_NOT_FOUND,
        progress_callback=on_source_event,
    )
    print("✅ [Stage 2] Kế hoạch:", version_result["plan"]["summary"])
    print("✅ [Stage 2] Kết quả:", version_result["summary"])

    # Xuất ngay Bảng 2: m1_download_stats.csv và m1_download_versions.csv từ manifest
    from src.scripts.audit_version_downloads import sync_download_stats_from_manifest
    stats_dest, v_dest = sync_download_stats_from_manifest(FAST_LOCAL_DATA)
    if stats_dest and stats_dest.is_file():
        print(f"📊 [Stage 2] Đã xuất Bảng 2: {stats_dest.name} ({len(pd.read_csv(stats_dest))} bài báo).")

tracker.download_time_sec = round(time.perf_counter() - t_stage2, 2)
tracker.sample_ram()


In [ ]:
# STAGE 3: CÀO TRÍCH DẪN SEMANTIC SCHOLAR (S2 BATCH API)
try:
    from src.crawl_references_module.raw_reference_collector import (
        collect_raw_references,
        publish_references,
        export_raw_reference_stats,
    )
except ModuleNotFoundError as e:
    missing_mod = getattr(e, "name", str(e))
    print(f"⚠️ Phát hiện thiếu module '{missing_mod}'. Đang tự động cài đặt...")
    import subprocess, sys, importlib
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "httpx>=0.27"], check=False)
    importlib.invalidate_caches()
    from src.crawl_references_module.raw_reference_collector import (
        collect_raw_references,
        publish_references,
        export_raw_reference_stats,
    )
    print("✅ Đã cài đặt xong thư viện và nạp module thành công!")

t_stage3 = time.perf_counter()

if STAGE_3_RAW_REFERENCES:
    eff_s2_interval = float(S2_MIN_INTERVAL) if "S2_MIN_INTERVAL" in globals() else 2.0
    print(f"🚀 [Stage 3] Cào trích dẫn Semantic Scholar (Batch size: {RAW_REF_BATCH_SIZE}, Interval: {eff_s2_interval}s, Fast Skip O(1))...")
    
    def on_ref_event(done, total, event):
        tracker.sample_ram()
        paper = event.get("arxiv_id", "")
        status = event.get("status", "")
        count = event.get("raw_refs_total", 0)
        if status == "ok":
            tracker.raw_refs_total += count
            
        pct = 100 * done / total if total else 100
        print(f"[Refs] {done}/{total} ({pct:.1f}%) | {paper} | {status} | {count} mục", flush=True)

    from src.crawl_references_module.s2_api_client import S2ApiClient
    active_s2_client = S2ApiClient(min_interval=eff_s2_interval)
    raw_ref_result = collect_raw_references(
        FAST_LOCAL_DATA,
        ids=target_ids,
        max_papers=0,
        batch_size=RAW_REF_BATCH_SIZE,
        dry_run=False,
        retry_errors=RETRY_ERRORS,
        retry_not_found=RETRY_NOT_FOUND,
        fast_skip=True, # Zero-second skip qua ledger
        client=active_s2_client,
        progress_callback=on_ref_event,
    )
    print("✅ [Stage 3] Kết quả:", raw_ref_result["summary"])
    if (FAST_LOCAL_DATA / "references_raw.jsonl").is_file():
        publish_references(FAST_LOCAL_DATA)

tracker.reference_time_sec = round(time.perf_counter() - t_stage3, 2)
tracker.sample_ram()


In [ ]:
# TỔNG KẾT PHIÊN, XUẤT CÁC BẢNG THỐNG KÊ (BẢNG 2, BẢNG 4) VÀ ĐỒNG BỘ DRIVE
import json, uuid, shutil
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display
from src import utils

# 1. Đợi đồng bộ các file còn lại trong queue sang Drive
print("⏳ Đang đồng bộ nốt các file raw trong hàng đợi sang Google Drive...")
sync_worker.queue.join()
print(f"✅ Đã đồng bộ {sync_worker.synced_count} versions sang Google Drive.")

# 2. XUẤT BẢNG 2: m1_download_stats.csv (và m1_download_versions.csv) TỪ MANIFEST
from src.scripts.audit_version_downloads import sync_download_stats_from_manifest
dl_stats_csv, dl_versions_csv = sync_download_stats_from_manifest(FAST_LOCAL_DATA)
if not dl_stats_csv or not dl_stats_csv.is_file():
    dl_stats_csv = FAST_LOCAL_DATA / "csv" / "m1_download_stats.csv"
    if not dl_stats_csv.is_file():
        pd.DataFrame(columns=[
            "arxiv_id", "download_status", "has_tex",
            "raw_size_bytes", "cleaned_size_bytes", "num_tex_files", "download_time_sec"
        ]).to_csv(dl_stats_csv, index=False)
print(f"✅ Đã xuất Bảng 2: {dl_stats_csv.name}")

# 3. Tính toán chính xác số lượng ID BÀI BÁO THỰC TẾ (không tính gộp version)
manifest_path = FAST_LOCAL_DATA / "versions_manifest.jsonl"
papers_path = FAST_LOCAL_DATA / "papers.jsonl"
ledger_path = FAST_LOCAL_DATA / "references_raw_ledger.jsonl"

completed_versions_by_paper = {}
if manifest_path.is_file():
    for row in utils.read_jsonl(manifest_path):
        aid = row.get("arxiv_id")
        if aid and row.get("status") in {"success", "pdf_only"}:
            completed_versions_by_paper.setdefault(aid, set()).add(row.get("version"))

paper_versions_expected = {}
if papers_path.is_file():
    for row in utils.read_jsonl(papers_path):
        aid = row.get("arxiv_id")
        if aid and isinstance(row.get("versions"), list):
            paper_versions_expected[aid] = set(row.get("versions"))

completed_refs_papers = set()
if ledger_path.is_file():
    for row in utils.read_jsonl(ledger_path):
        aid = row.get("arxiv_id")
        if aid and row.get("status") == "ok":
            completed_refs_papers.add(aid)

session_unique_papers_completed = 0
for aid in target_ids:
    can_aid = utils.canonical_arxiv_id(aid)
    norm_aid = utils.normalize_arxiv_id(aid)
    key = can_aid if can_aid in paper_versions_expected else norm_aid
    
    expected_vs = paper_versions_expected.get(key, set())
    got_vs = completed_versions_by_paper.get(key, completed_versions_by_paper.get(can_aid, set()))
    
    has_versions = bool(got_vs and (not expected_vs or expected_vs.issubset(got_vs)))
    has_refs = (key in completed_refs_papers or can_aid in completed_refs_papers or norm_aid in completed_refs_papers)
    
    if has_versions:
        session_unique_papers_completed += 1

total_wall_time = round(time.time() - tracker.start_time, 2)
peak_ram_mb = round(tracker.peak_ram_bytes / (1024 * 1024), 2)
disk_used_mb = round(sum(f.stat().st_size for f in FAST_LOCAL_DATA.glob("**/*") if f.is_file()) / (1024 * 1024), 2)

avg_raw_size = round(tracker.total_raw_bytes / max(1, tracker.downloaded_versions_count), 2)
avg_clean_size = round(tracker.total_cleaned_bytes / max(1, tracker.downloaded_versions_count), 2)
size_saving_pct = round(100 * (1 - (avg_clean_size / max(1, avg_raw_size))), 1) if avg_raw_size else 0.0
avg_refs_per_paper = round(tracker.raw_refs_total / max(1, session_unique_papers_completed), 1)
avg_sec_per_paper = round(total_wall_time / max(1, session_unique_papers_completed), 2)

# 4. XUẤT BẢNG 4: m1_version_pipeline_benchmark.csv & m1_system_benchmark.csv
bench_csv_path = FAST_LOCAL_DATA / "csv" / "m1_version_pipeline_benchmark.csv"
legacy_bench_path = FAST_LOCAL_DATA / "csv" / "m1_system_benchmark.csv"

if "benchmark_tracker" in globals() and benchmark_tracker is not None:
    df_benchmark = benchmark_tracker.stop(
        total_papers=len(target_ids),
        csv_path=bench_csv_path,
        final_storage_mb=disk_used_mb,
    )
else:
    avg_ram = round(sum(tracker.ram_samples) / max(1, len(tracker.ram_samples)) / (1024 * 1024), 2) if hasattr(tracker, "ram_samples") and tracker.ram_samples else peak_ram_mb
    peak_disk_calc = round(disk_used_mb + (tracker.total_raw_bytes / (1024 * 1024)), 2) if tracker.total_raw_bytes else disk_used_mb
    bench_record = {
        "total_papers": len(target_ids),
        "total_wall_time_sec": total_wall_time,
        "peak_ram_mb": peak_ram_mb,
        "average_ram_mb": avg_ram,
        "peak_disk_mb": peak_disk_calc,
        "final_storage_mb": disk_used_mb,
    }
    df_benchmark = pd.DataFrame([bench_record])
    df_benchmark.to_csv(bench_csv_path, index=False)

# Lưu cả vào m1_system_benchmark.csv để tương thích hoàn toàn
df_benchmark.to_csv(legacy_bench_path, index=False)
print(f"✅ Đã xuất Bảng 4: {bench_csv_path.name}")

# 5. Xuất file crawl_session_stats.json & .csv
session_stats = {
    "session_id": str(uuid.uuid4())[:8],
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "start_id": start_id,
    "end_id": end_id,
    "worker_index": WORKER_INDEX if ENABLE_SHARDING else 0,
    "total_workers": TOTAL_WORKERS if ENABLE_SHARDING else 1,
    "target_papers_assigned": len(target_ids),
    "session_crawled_unique_papers": session_unique_papers_completed,
    "total_versions_downloaded": tracker.downloaded_versions_count,
    "total_pdf_only_versions": tracker.pdf_only_versions_count,
    "total_raw_references_collected": tracker.raw_refs_total,
    "avg_references_per_paper": avg_refs_per_paper,
    "avg_raw_size_bytes": avg_raw_size,
    "avg_cleaned_size_bytes": avg_clean_size,
    "size_reduction_pct": size_saving_pct,
    "discovery_time_sec": tracker.discovery_time_sec,
    "download_time_sec": tracker.download_time_sec,
    "reference_time_sec": tracker.reference_time_sec,
    "total_wall_time_sec": total_wall_time,
    "avg_sec_per_paper": avg_sec_per_paper,
    "peak_ram_mb": peak_ram_mb,
    "local_disk_used_mb": disk_used_mb,
}

stats_json_path = FAST_LOCAL_DATA / "csv" / "crawl_session_stats.json"
stats_csv_path = FAST_LOCAL_DATA / "csv" / "crawl_session_stats.csv"

stats_json_path.write_text(json.dumps(session_stats, indent=2, ensure_ascii=False), encoding="utf-8")
pd.DataFrame([session_stats]).to_csv(stats_csv_path, index=False)

# 6. Đồng bộ toàn bộ metadata JSONL và CSV lên Google Drive
if PERSISTENT_DRIVE_DATA.is_dir():
    print("⏳ Đang cập nhật toàn bộ metadata JSONL và CSV lên Google Drive...")
    for f in FAST_LOCAL_DATA.glob("*.jsonl*"):
        shutil.copy2(f, PERSISTENT_DRIVE_DATA / f.name)
    (PERSISTENT_DRIVE_DATA / "csv").mkdir(parents=True, exist_ok=True)
    for f in (FAST_LOCAL_DATA / "csv").glob("*.csv"):
        shutil.copy2(f, PERSISTENT_DRIVE_DATA / "csv" / f.name)
    for f in (FAST_LOCAL_DATA / "csv").glob("*.json"):
        shutil.copy2(f, PERSISTENT_DRIVE_DATA / "csv" / f.name)
    print("✅ Đã đồng bộ hoàn tất toàn bộ file CSV & JSONL sang Drive.")

# 7. Hiển thị trực quan các bảng kết quả
print("\n" + "=" * 70)
print("📊 BẢNG 2: m1_download_stats.csv (Thống kê tải LaTeX Source)")
print("=" * 70)
if dl_stats_csv.is_file():
    display(pd.read_csv(dl_stats_csv, dtype={"arxiv_id": str}).head(10))

print("\n" + "=" * 70)
print("📊 BẢNG 4: m1_version_pipeline_benchmark.csv (Hiệu năng phần cứng hệ thống)")
print("=" * 70)
display(df_benchmark)

print("\n" + "=" * 70)
print("📊 BÁO CÁO TỔNG KẾT PHIÊN CÀO DỮ LIỆU MILESTONE 1")
print("=" * 70)

display_df = pd.DataFrame([
    {"Chỉ số (Metric)": "Số bài báo được giao (Target Unique Papers)", "Giá trị": len(target_ids)},
    {"Chỉ số (Metric)": "Số bài báo HOÀN THÀNH THỰC TẾ (Unique Paper IDs)", "Giá trị": session_unique_papers_completed},
    {"Chỉ số (Metric)": "Tổng số phiên bản TeX tải về (Versions)", "Giá trị": tracker.downloaded_versions_count},
    {"Chỉ số (Metric)": "Số phiên bản PDF-only (không có TeX)", "Giá trị": tracker.pdf_only_versions_count},
    {"Chỉ số (Metric)": "Tổng số trích dẫn cào được (Raw References)", "Giá trị": tracker.raw_refs_total},
    {"Chỉ số (Metric)": "Số trích dẫn trung bình / bài (Refs/Paper)", "Giá trị": avg_refs_per_paper},
    {"Chỉ số (Metric)": "Dung lượng thô TB / bài (Raw size bytes)", "Giá trị": f"{avg_raw_size:,.0f} B"},
    {"Chỉ số (Metric)": "Dung lượng sạch TB sau khi xóa ảnh (Cleaned size)", "Giá trị": f"{avg_clean_size:,.0f} B"},
    {"Chỉ số (Metric)": "Tỉ lệ tiết kiệm đĩa sau khi xóa ảnh (%)", "Giá trị": f"{size_saving_pct}%"},
    {"Chỉ số (Metric)": "Tổng thời gian thực thi (Total Wall Time)", "Giá trị": f"{total_wall_time} giây ({total_wall_time/60:.1f} phút)"},
    {"Chỉ số (Metric)": "Thời gian trung bình / bài báo (Avg sec/paper)", "Giá trị": f"{avg_sec_per_paper} giây"},
    {"Chỉ số (Metric)": "RAM đỉnh tiêu thụ (Peak RAM)", "Giá trị": f"{peak_ram_mb} MB"},
    {"Chỉ số (Metric)": "Dung lượng đĩa SSD sử dụng (Local Disk)", "Giá trị": f"{disk_used_mb} MB"},
    {"Chỉ số (Metric)": "Bảng 2 xuất ra", "Giá trị": "m1_download_stats.csv"},
    {"Chỉ số (Metric)": "Bảng 4 xuất ra", "Giá trị": "m1_version_pipeline_benchmark.csv"},
])

display(display_df)
print(f"\n🎉 Hoàn thành cào: {session_unique_papers_completed}/{len(target_ids)} bài báo.")
print(f"📁 Toàn bộ file kết quả đã sẵn sàng tại: {FAST_LOCAL_DATA / 'csv'} và Google Drive.")
